In [1]:
import pandas as pd

In [7]:
df = pd.read_excel("Movies_List_2026-2027.xls")

In [8]:
df

,S.No.,Movie/Content Name,Movie /Content (Original Language),Language,Type of Content,Availability,Col B&W,Year 30,No. of Episodes,Length,...,Agreement Status,Size,Format,Audio bitrate,Audio codec,Video bitrate,Video codec,Framerate,Path,-
0,-,143 (2017),(143 - TAMIL),Hindi Dubed,Movie,Yes,COL,2017.0,-,01:49:25,...,Yes,1285.20 MB,Matroska,NaN,AAC LC,NaN,V_VP9,25.000 fps,U:\MOVIE MASTER\143 (2017)\143 (2017).mp4,-
1,-,2035 Steel Superstorm,-,Hindi,Eng.to Hindi,Yes,COL,2013.0,-,01:32:20,...,Yes,3016.44 MB,MPEG-4,128 kbps,AAC LC,4434 kbps,av01,23.976 fps,U:\MOVIE MASTER\2035 Steel Superstorm\2035 Ste...,-
2,-,26Th July At Barista (2008),-,Hindi,Movie,Yes,COL,2008.0,-,01:19:26,...,Yes,343.40 MB,MPEG-4,128 kbps,AAC LC,473 kbps,AVC,25.000 fps,U:\MOVIE MASTER\26th July At Barista (2008)\26...,-
3,-,27th December 1987 Final Match (2016),-,Hindi,Movie,Yes,COL,2016.0,-,01:59:31,...,Yes,1869.09 MB,Matroska,NaN,AAC LC,NaN,V_VP9,25.000 fps,U:\MOVIE MASTER\27th December 1987 Final Match...,-
4,-,3 Deewarein (2003),-,Hindi,Movie,Yes,COL,2003.0,-,02:00:46,...,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5039,NaN,Zulm Ko Jala Doonga (1988),NaN,NaN,NaN,NaN,NaN,NaN,NaN,02:04:34,...,NaN,12463.38 MB,MPEG-4,317 kbps,AAC LC,13667 kbps,AVC,25.000 fps,U:\MOVIE MASTER\Zulm Ko Jala Doonga (1988)\Zul...,-
5040,-,Zulmi (1999),-,Hindi,Movie,Yes,COL,1999.0,-,0.098704,...,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
5041,NaN,Zulmi (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,02:21:34,...,NaN,1435.98 MB,MPEG-4,126 kbps,AAC LC,1289 kbps,AVC,25.000 fps,U:\MOVIE MASTER\Zulmi (1999)\Zulmi (1999).MP4,-
5042,-,Zulm-O-Sitam (1998),-,Hindi,Movie,Yes,COL,1998.0,-,0.090046,...,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-


In [9]:
import numpy as np

# 1. Load the data
# df = pd.read_excel("Movies_List_2026-2027.xls")

# 2. Clean up placeholder dashes so they are recognized as missing data
df.replace('-', np.nan, inplace=True)

# 3. Define a custom function to merge the overlapping columns
def combine_rows(series):
    # Drop all missing values from the grouped series
    valid_data = series.dropna()
    # Return the upper row's data if it exists; otherwise fallback, or return NaN
    return valid_data.iloc[0] if not valid_data.empty else np.nan

# 4. Group by the movie name and apply our custom rule
cleaned_df = df.groupby('Movie/Content Name', as_index=False).agg(combine_rows)

print("Original rows:", df.shape[0])
print("Cleaned rows:", cleaned_df.shape[0])

C:\Users\Harshraj\AppData\Local\Temp\ipykernel_14736\676963763.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('-', np.nan, inplace=True)


Original rows: 5044
Cleaned rows: 3306


In [10]:
# 1. Clean the 'Size' column: remove ' MB' and convert to float
cleaned_df['Size_MB'] = (
    cleaned_df['Size']
    .astype(str)
    .str.replace(' MB', '', regex=False)
    .str.replace(',', '') # Just in case any numbers have commas
    .astype(float)
)

# 2. Clean the 'Framerate' column: remove ' fps' and convert to float
cleaned_df['Framerate_fps'] = (
    cleaned_df['Framerate']
    .astype(str)
    .str.replace(' fps', '', regex=False)
    .astype(float)
)

# 3. Convert date columns to proper datetime objects
date_cols = ['Commence Date', 'Expiry Date']
for col in date_cols:
    cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

# Drop the old text-based columns to keep the dataframe tidy
cleaned_df.drop(columns=['Size', 'Framerate'], inplace=True)

# Verify the changes
print(cleaned_df[['Movie/Content Name', 'Size_MB', 'Framerate_fps', 'Commence Date']].head())
print("\nNew Data Types:")
print(cleaned_df[['Size_MB', 'Framerate_fps', 'Commence Date', 'Expiry Date']].dtypes)

               Movie/Content Name  Size_MB  Framerate_fps Commence Date
0                      143 (2017)  1285.20           25.0    2026-04-01
1              16 December (2002)   721.70           25.0           NaT
2  18.11 A Code Of Secrecy (2014)  6121.61            NaN           NaT
3  1811 A Code Of Secrecy  (2014)      NaN            NaN    2026-04-01
4                   1920   (2008)  1251.66           25.0           NaT

New Data Types:
Size_MB                 float64
Framerate_fps           float64
Commence Date    datetime64[ns]
Expiry Date      datetime64[ns]
dtype: object


In [11]:
# Export the cleaned dataframe to a new Excel file
cleaned_df.to_excel("Cleaned_Movies_List_Final.xlsx", index=False)

# Alternatively, export it as a CSV (often preferred for faster loading in data science projects)
cleaned_df.to_csv("Cleaned_Movies_List_Final.csv", index=False)

print("Export complete! Check your folder for the new files.")

Export complete! Check your folder for the new files.


In [12]:
import pandas as pd
import numpy as np

# 1. Load the original data
filepath = "Movies_List_2026-2027.xls"
df = pd.read_excel(filepath)

# Save the original column order to re-apply later
original_columns = df.columns.tolist()

# 2. Clean spaces and missing values
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df.replace(['-', '', ' ', 'NA', 'N/A'], np.nan, inplace=True)

# 3. Define the merge function
def combine_rows(series):
    valid_data = series.dropna()
    return valid_data.iloc[0] if not valid_data.empty else np.nan

# 4. Group and aggregate (CRITICAL: sort=False keeps original row order)
cleaned_df = df.groupby('Movie/Content Name', as_index=False, sort=False).agg(combine_rows)

# 5. Restore the exact original column order
cleaned_df = cleaned_df[original_columns]

# 6. Export to Excel using openpyxl engine
output_file = "Cleaned_Movies_List_Formatted.xlsx"

# Using pandas ExcelWriter to add basic formatting
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    cleaned_df.to_excel(writer, index=False, sheet_name='Green')
    
    # Access the workbook and active sheet to adjust column widths
    worksheet = writer.sheets['Green']
    for column_cells in worksheet.columns:
        # Determine the maximum length of data in the column
        length = max(len(str(cell.value)) for cell in column_cells)
        # Set the column width based on the length
        worksheet.column_dimensions[column_cells[0].column_letter].width = length + 2

print(f"Done! Open '{output_file}' to see your perfectly structured data.")

C:\Users\Harshraj\AppData\Local\Temp\ipykernel_14736\1628050503.py:12: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
C:\Users\Harshraj\AppData\Local\Temp\ipykernel_14736\1628050503.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['-', '', ' ', 'NA', 'N/A'], np.nan, inplace=True)


Done! Open 'Cleaned_Movies_List_Formatted.xlsx' to see your perfectly structured data.
